In [1]:
PATH_TO_EXPLANATION_IT = "../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/"
PATH_TO_EVALUATION_IT = "../data/evaluation/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/"
PATH_TO_TEMPLATE = "../data/evaluation/survey_template.txt"

MAP_LABELS_IT = {"0": "Released", "1": "NotReleased"}
MAP_LABELS_EN = {"0": "negative", "1": "positive"}



In [2]:
import os

os.makedirs(PATH_TO_EXPLANATION_IT, exist_ok=True)
os.makedirs(PATH_TO_EVALUATION_IT, exist_ok=True)

In [3]:
import glob

explanations_files = glob.glob(PATH_TO_EXPLANATION_IT + "*.json")
explanations_files

['../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/650.json',
 '../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/676.json',
 '../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/687.json',
 '../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/704.json',
 '../data/explanations/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027/710.json']

In [4]:
import os
import json
from pathlib import Path

TEMPLATE_EXPLANATION = open(PATH_TO_TEMPLATE, "r").read().strip()


def process_explanation(target: Path, output_folder: Path):
    with open(target, "r") as fp:
        json_explanation = json.load(fp)
    print(json.dumps(json_explanation, indent=3))

    os.makedirs(output_folder, exist_ok=True)

    print("File:", target)
    doc_id = target.stem
    y_true = json_explanation.get('y_pred')
    y_pred = json_explanation.get('y_test')
    content = json_explanation.get('original_content')

    template = str(TEMPLATE_EXPLANATION)

    template = template.replace("@DOC_ID", str(doc_id))
    template = template.replace("@LABEL", MAP_LABELS_IT[str(y_true)])
    template = template.replace("@PREDICTION", MAP_LABELS_IT[str(y_pred)])
    template = template.replace("@TEXT", str(content).strip())
    template = template.replace("@GRAPH_PDF_FILE", f"graph_{doc_id}.pdf")

    explanation_per_hypernode = json_explanation["explanation"]
    for hyper_node_explanation_id in list(explanation_per_hypernode)[:2]:
        template_explanation = str(template)  # Copy for this specific explanation

        print("Hyper node: " + hyper_node_explanation_id)
        explanation_content = explanation_per_hypernode[hyper_node_explanation_id]

        words_l0 = explanation_content["words_l0"]
        cg_methods = explanation_content["words_cg_methods"]

        llm_search = cg_methods["llm_search"]
        semantic_search_l0 = cg_methods["semantic_search_l0"]
        semantic_search_l1 = cg_methods["semantic_search_l1"]
        top_l0 = cg_methods["top_l0"]

        explanation_object = """
### Abstract Node ID
@HYPERNODE_ID

### Words highlighted from Layer 0 assigned to this abstract node in Layer 1
@WORDS_L0

### Concept grounding for abstract node in Layer 1

Method 1:
@LLM_SEARCH

Method 2:
@SEMANTIC_SEARCH_L1

Method 3:
@SEMANTIC_SEARCH_L0

Method 4:
@TOP_L0
        """.strip()

        explanation_object = explanation_object.replace("@HYPERNODE_ID", hyper_node_explanation_id)
        explanation_object = explanation_object.replace("@WORDS_L0", ", ".join(words_l0))

        # Method 1
        explanation_object = explanation_object.replace("@LLM_SEARCH", ", ".join(llm_search))
        # Method 2
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L1", ", ".join(semantic_search_l1))
        # Method 3
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L0", ", ".join(semantic_search_l0))
        # Method 4
        explanation_object = explanation_object.replace("@TOP_L0", ", ".join(top_l0))

        template_explanation = template_explanation.replace("@EXPLANATION_OBJECT", explanation_object)

        output_file = Path(
            output_folder) / f"Explanation_Doc-{int(doc_id):03d}_Hypernode_{int(hyper_node_explanation_id):03d}.txt"
        with open(output_file, "w") as fp:
            fp.write(template_explanation)

    # Doc ID

    # Label
    # Prediction

    # Questions
    # 1. Is the prediction correct?


In [5]:
for explanation_file in explanations_files[:3]:
    process_explanation(Path(explanation_file), PATH_TO_EVALUATION_IT)

{
   "explanation": {
      "11": {
         "words_l0": [
            "n",
            "disp",
            "lett",
            "novembre",
            "catanzaro"
         ],
         "words_cg_methods": {
            "top_l0": [
               "n"
            ],
            "llm_search": [
               "diritto",
               "giustizia",
               "processo"
            ],
            "semantic_search_l0": [
               "comma l n",
               "l n comma",
               "n comma al"
            ],
            "semantic_search_l1": [
               "plurisoggettiva e tuttavia",
               "sindacabile conseguenza con",
               "solo di sindacabilit\u00e0"
            ]
         }
      },
      "4": {
         "words_l0": [
            "n",
            "catanzaro",
            "disp",
            "novembre",
            "potuto"
         ],
         "words_cg_methods": {
            "top_l0": [
               "n"
            ],
            "llm_search": [
